In [ ]:
# --- project bootstrap ---
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(ROOT)
# -------------------------


# Immortal chiral edge numerics (Markov monitoring)

This notebook builds the two PRL-style figures described in the request. It uses:
- `classA_U1FGTN` for the chiral edge (domain-wall Chern insulator).
- The Markov-channel update implied by `run_markov_channel`.

If a `CI_Lindblad_DW` class is available in your environment, the non-chiral comparison
will use it; otherwise the notebook falls back to a simple 1D Gaussian chain model.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import importlib

# Thread caps (match style used in exact_CI_DW_testing.ipynb)
os.environ.setdefault("OMP_NUM_THREADS", "8")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "8")
os.environ.setdefault("MKL_NUM_THREADS", "8")
os.environ.setdefault("NUMEXPR_MAX_THREADS", "8")

import fgtn.classA_U1FGTN as classA_U1FGTN
importlib.reload(classA_U1FGTN)
from fgtn.classA_U1FGTN import classA_U1FGTN
plt.rcParams.update({"font.size": 12})


In [ ]:
# Optional: non-chiral class (if you have it locally)
have_ci_lindblad = False
CI_Lindblad_DW = None
try:
    from CI_Lindblad_DW import CI_Lindblad_DW  # noqa: F401
    have_ci_lindblad = True
except Exception:
    have_ci_lindblad = False

print(f"CI_Lindblad_DW available: {have_ci_lindblad}")


## Helper functions
These implement entanglement extraction, edge-mode indexing, and correlators in the
Gaussian (covariance-matrix) framework.


In [ ]:
def top_layer_from_full(G_full):
    G_full = np.asarray(G_full, dtype=np.complex128)
    if G_full.ndim != 2:
        raise ValueError("Expected 2D covariance matrix.")
    # If full layer is passed (top+bottom), take top-left block.
    if G_full.shape[0] % 2 == 0:
        Nlayer = G_full.shape[0] // 2
        if G_full.shape[1] == G_full.shape[0]:
            return G_full[:Nlayer, :Nlayer]
    return G_full


def G2pt_from_G(G_top):
    G_top = np.asarray(G_top, dtype=np.complex128)
    N = G_top.shape[0]
    return 0.5 * (np.eye(N, dtype=np.complex128) + G_top)


def site_index(mu, x, y, Nx):
    # Top-layer indexing convention used in classA_U1FGTN
    return int(mu + 2 * x + 2 * Nx * y)


def entanglement_edge_segment(model, G_top, ell, x_window, y0=0):
    s_map = model.entanglement_contour(G_top, model.Nx, model.Ny)
    y1 = min(model.Ny, y0 + int(ell))
    if y1 <= y0:
        return 0.0
    return np.sum(s_map[np.ix_(x_window, np.arange(y0, y1))])


def fit_central_charge(ell_list, S_list, L):
    ell = np.asarray(ell_list, dtype=float)
    S = np.asarray(S_list, dtype=float)
    x = np.log((L / np.pi) * np.sin(np.pi * ell / L))
    m, b = np.polyfit(x, S, 1)
    c_eff = 3.0 * m
    return c_eff, (m, b, x)


def edge_x_window(model, half_width=1):
    # Use DW_loc if present; otherwise center.
    if hasattr(model, "DW_loc") and model.DW_loc:
        x_center = int(round(np.mean(model.DW_loc)))
    else:
        x_center = model.Nx // 2
    x0 = max(0, x_center - half_width)
    x1 = min(model.Nx, x_center + half_width + 1)
    return np.arange(x0, x1, dtype=int)


def fermion_correlator_along_edge(G_top, Nx, Ny, x_fixed, y0, mu=0, max_r=None):
    G2 = G2pt_from_G(G_top)
    if max_r is None:
        max_r = Ny - y0 - 1
    idx0 = site_index(mu, x_fixed, y0, Nx)
    vals = []
    for r in range(1, max_r + 1):
        y = y0 + r
        idx = site_index(mu, x_fixed, y, Nx)
        vals.append(G2[idx, idx0])
    return np.array(vals)


def density_connected_along_edge(G_top, Nx, Ny, x_fixed, y0, max_r=None):
    G2 = G2pt_from_G(G_top)
    if max_r is None:
        max_r = Ny - y0 - 1

    mu_list = [0, 1]
    idx0 = [site_index(mu, x_fixed, y0, Nx) for mu in mu_list]
    vals = []
    for r in range(1, max_r + 1):
        y = y0 + r
        idx = [site_index(mu, x_fixed, y, Nx) for mu in mu_list]
        # Connected density correlation for i != j
        block = G2[np.ix_(idx, idx0)]
        vals.append(-np.sum(np.abs(block) ** 2))
    return np.array(vals)


## Figure 1: Entanglement scaling vs measurement strength
This section constructs the chiral-edge data and extracts the effective central charge.


In [ ]:
# --- Chiral system parameters ---
Nx = 16
Ny = 40
alpha_1, alpha_2 = 30, 1
periodic = True

# Monitoring parameters
cycles = 40
n_a = 0.5
# Map measurement strength gamma -> p in run_markov_channel
# Interpret p ~= gamma * dt, clipped to (0, 1].
dt = 0.2

gamma_list = [0.0, 0.2, 0.5, 1.0, 2.0]

def gamma_to_p(gamma):
    if gamma <= 0:
        return 0.0
    return float(np.clip(gamma * dt, 1e-6, 1.0))

# Build chiral model and GS covariance
chiral = classA_U1FGTN(Nx, Ny, DW=True, nshell=None, alpha_1=alpha_1, alpha_2=alpha_2)
G_gs = chiral.G_CI_domain_wall(periodic=periodic)
chiral.G0 = G_gs.copy()

x_window = edge_x_window(chiral, half_width=1)
ell_list = np.arange(2, Ny // 2)

chiral_S = {}
chiral_c_eff = {}
chiral_Gsteady = {}

for gamma in gamma_list:
    p = gamma_to_p(gamma)
    if p <= 0.0:
        G_top = G_gs.copy()
    else:
        res = chiral.run_markov_channel(
            G_history=False,
            cycles=cycles,
            p=p,
            init_mode="custom",
            n_a=n_a,
            progress=True,
            save=False,
            sequence="snake_y",
        )
        G_top = res["G_final_avg"]
    chiral_Gsteady[gamma] = G_top

    S_list = [entanglement_edge_segment(chiral, G_top, ell, x_window, y0=0) for ell in ell_list]
    c_eff, _ = fit_central_charge(ell_list, S_list, Ny)
    chiral_S[gamma] = S_list
    chiral_c_eff[gamma] = c_eff

print("Chiral c_eff:", chiral_c_eff)


In [ ]:
# Panel A: S(ell) vs log(sin) for the chiral edge
plt.figure(figsize=(7, 5))
for gamma in gamma_list:
    S_list = np.array(chiral_S[gamma])
    x = np.log((Ny / np.pi) * np.sin(np.pi * ell_list / Ny))
    plt.plot(x, S_list, marker="o", ms=3, lw=1.2, label=fr"$\gamma={gamma}$")

plt.xlabel(r"$\log\left(\frac{L}{\pi}\sin(\pi \ell / L)\right)$")
plt.ylabel(r"$S(\ell)$")
plt.title("Chiral edge: entanglement scaling vs measurement strength")
plt.grid(alpha=0.3)
plt.legend(loc="best", fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# Panel B: c_eff vs gamma (chiral)
plt.figure(figsize=(6, 4))
plt.plot(gamma_list, [chiral_c_eff[g] for g in gamma_list], "o-", lw=1.5)
plt.axhline(1.0, color="k", lw=1, ls="--", alpha=0.6)
plt.xlabel(r"Measurement strength $\gamma$")
plt.ylabel(r"$c_{\mathrm{eff}}$")
plt.title("Chiral edge: central charge is robust")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### Non-chiral comparison
If `CI_Lindblad_DW` is available, use it here. Otherwise, we fall back to a minimal
1D Gaussian chain with density dephasing as a placeholder for the non-chiral wire.


In [ ]:
def free_fermion_chain_covariance(L, filling=0.5):
    # 1D tight-binding chain with periodic boundary conditions
    k = np.fft.fftfreq(L, d=1.0)
    # sort k to match energies
    k = 2 * np.pi * k
    eps = -2.0 * np.cos(k)
    order = np.argsort(eps)
    occ = np.zeros(L, dtype=bool)
    occ[order[: int(round(filling * L))]] = True
    # Build correlation matrix in position space
    U = np.exp(1j * np.outer(np.arange(L), k)) / np.sqrt(L)
    C = U[:, occ] @ U[:, occ].conj().T
    return C


def dephasing_step(C, H, gamma, dt):
    # dC/dt = -i[H,C] - gamma*(C - diag(C))
    comm = H @ C - C @ H
    C = C - 1j * dt * comm
    C = C - gamma * dt * (C - np.diag(np.diag(C)))
    C = 0.5 * (C + C.conj().T)
    return C


def evolve_dephasing_chain(L, gamma, T, dt, filling=0.5):
    # Hamiltonian for 1D hopping
    H = np.zeros((L, L), dtype=np.complex128)
    for i in range(L):
        H[i, (i + 1) % L] = -1.0
        H[(i + 1) % L, i] = -1.0
    C = free_fermion_chain_covariance(L, filling=filling)
    steps = int(round(T / dt))
    for _ in range(steps):
        C = dephasing_step(C, H, gamma, dt)
    return C


def chain_entanglement_entropy(C, ell):
    # Gaussian entropy from correlation matrix restriction
    idx = np.arange(ell)
    Csub = C[np.ix_(idx, idx)]
    evals = np.clip(np.real_if_close(np.linalg.eigvalsh(Csub)), 1e-12, 1 - 1e-12)
    S = -np.sum(evals * np.log(evals) + (1 - evals) * np.log(1 - evals))
    return S

nonchiral_c_eff = {}
nonchiral_S = {}

L_chain = 128
ell_chain = np.arange(2, L_chain // 2)
T_chain = 40

if have_ci_lindblad:
    print("TODO: plug CI_Lindblad_DW steady-state covariance here.")
else:
    for gamma in gamma_list:
        C = evolve_dephasing_chain(L_chain, gamma, T_chain, dt, filling=0.5)
        S_list = [chain_entanglement_entropy(C, ell) for ell in ell_chain]
        # CFT fit using same log-sin variable
        x = np.log((L_chain / np.pi) * np.sin(np.pi * ell_chain / L_chain))
        m, b = np.polyfit(x, S_list, 1)
        nonchiral_c_eff[gamma] = 3.0 * m
        nonchiral_S[gamma] = S_list

print("Non-chiral c_eff:", nonchiral_c_eff)


In [ ]:
# Panel B (combined): chiral vs non-chiral
plt.figure(figsize=(6, 4))
plt.plot(gamma_list, [chiral_c_eff[g] for g in gamma_list], "o-", label="chiral edge")
if nonchiral_c_eff:
    plt.plot(gamma_list, [nonchiral_c_eff[g] for g in gamma_list], "s-", label="non-chiral chain")
plt.axhline(1.0, color="k", lw=1, ls="--", alpha=0.6)
plt.xlabel(r"Measurement strength $\gamma$")
plt.ylabel(r"$c_{\mathrm{eff}}$")
plt.title("Effective central charge vs monitoring")
plt.grid(alpha=0.3)
plt.legend(loc="best", fontsize=9)
plt.tight_layout()
plt.show()


## Figure 2: Orthogonal-metal signature
We compare single-particle vs density correlations along the edge.


In [ ]:
# Choose two measurement strengths to compare
probe_gammas = [0.0, 1.0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for gamma in probe_gammas:
    G_top = chiral_Gsteady[gamma]
    x_fixed = int(np.mean(x_window))
    y0 = 0
    max_r = Ny // 2

    Gpsi = fermion_correlator_along_edge(G_top, Nx, Ny, x_fixed, y0, mu=0, max_r=max_r)
    Crho = density_connected_along_edge(G_top, Nx, Ny, x_fixed, y0, max_r=max_r)

    r = np.arange(1, max_r + 1)
    axes[0].plot(r, np.log(np.abs(Gpsi) + 1e-16), lw=1.5, label=fr"$\gamma={gamma}$")
    axes[1].plot(r, np.log(np.abs(Crho) + 1e-16), lw=1.5, label=fr"$\gamma={gamma}$")

axes[0].set_xlabel(r"$x$")
axes[0].set_ylabel(r"$\log |G_\psi(x)|$")
axes[0].set_title("Fermion correlator (edge)")
axes[0].grid(alpha=0.3)
axes[0].legend(loc="best", fontsize=9)

axes[1].set_xlabel(r"$x$")
axes[1].set_ylabel(r"$\log |C_\rho(x)|$")
axes[1].set_title("Density correlator (edge)")
axes[1].grid(alpha=0.3)
axes[1].legend(loc="best", fontsize=9)

plt.tight_layout()
plt.show()
